# Pydantic notes : 
- youtube link: https://www.youtube.com/watch?v=M81pfi64eeM&list=PL1pUCNO-yV2nJGCS4O0_rJUJbJMEPfsvk
- made on nov 2025 

### what is do ? : 
> it make sure that the data comming in the DB meet certain expectaion such as type and format of the data 
- its usage: 
    - Fast api is used this for validataing the data input from api requests 
    - used in data processing tools 
    - used where the data go to a system 
    - pydantic AI is used for AI tools 
- why don't we write type of data on out own? 

### mannual data type checking looks alike 



In [ ]:
def create_user(username, email, age):
    if not isinstance(username, str):
        raise TypeError('username must be a string')
    if not isinstance(email, str):
        raise TypeError('Email must be a string')
    if not isinstance(age, int):
        raise TypeError('Age must be an integer')
    return {'username': username, 'email': email, 'age':age}

user1 = create_user('chikoo', "chikoo@gmail.com", 34)
print(user1)

user2 = create_user('chotu', None, 'old')# different data 

# gives TypeError: Email must be a string
# after fixing it it will tell about next error means it tells about one error at a time
# so make it efficient we can use pydantic 

print(user2)



{'username': 'chikoo', 'email': 'chikoo@gmail.com', 'age': 34}


TypeError: Email must be a string

## using pydantic we can remove this problem

In [ ]:


from pydantic import BaseModel
class User(BaseModel):
    username: str
    email: str
    age: int 
user1 = User(username = 'codu', email='codu@gmail.com', age = 25)
print(user1)
# it give all the error at a same type 
#ValidationError: 2 validation errors for User
# email; age
user2 = User(username = 'hima', email =None, age = 'old')
print(user2)


username='codu' email='codu@gmail.com' age=25


ValidationError: 2 validation errors for User
email
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='old', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [ ]:
from pydantic import BaseModel, ValidationError
from datetime import datetime, UTC
from typing import Literal, Annotated

class User(BaseModel):
    uid: Annotated[int, Field(gt=0)] # its a way to add meta data to our constraints, eg uid is greated than 0. 

    username: str
    email : str 
    bio: str = ''
    verified_at: datetime | None = None
    is_active: bool = True
    age : Annotated[int, Field(ge=13,le=130)]
    full_name: str | None  = None # we making sure that full name should be string but in next we make its value none  & none is not a string, so for making it correct we use 'or' | operator(pipe ).


# lets create a user 

user = User(
    uid = 22,
    username='funcho',
    email = 'fucho@gmail.com',

    age = 18,
    bio ='fucked up',
)
# empty user is not valid because we dont make some default valuse there for user 

# we can add or change it as it is python code 

user.bio = '123'

print(user.bio)



# we can add or change it as it is python code 
# for saving the our model in  dict we can use method : model_dump()

print(user.model_dump())

# for json we can use model_dump_json()
print(user.model_dump_json())


123
{'uid': 22, 'username': 'funcho', 'email': 'fucho@gmail.com', 'bio': '123', 'verified_at': None, 'is_active': True, 'full_name': None}
{"uid":22,"username":"funcho","email":"fucho@gmail.com","bio":"123","verified_at":null,"is_active":true,"full_name":null}


In [ ]:
# if we need to make sure the user is registered in proper format we can implement the exceptiom

try : 


    user = User(
        uid =' 22', # error but not shown in the errors down 
        # it is because pydantic has type conversion enable by default and it sometime convert the suitatble type coversion , which has less memory usage. 
        # we have to make sure the size of the data 
        username=None,#  error
        email = 'fucho@gmail.com',

        age = 18,
        bio ='fucked up',
    )
except ValidationError as e: 
    print(e)

    # by this we can handle the error

1 validation error for User
username
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


# lets use this user clas in some example a


In [23]:
from pydantic import Field
from functools import partial #it will help us prefill some arguements to the funciton and return an unexcuted funcition . in this new fuction some arguemnts are prefilled.
from typing import Literal, Annotated
class BlogPost(BaseModel):
    title:str
    content: str 
    author_id : str |int
    view_count : int = 0
    is_published: bool = False
    tags: list[str] = Field(default_factory=list)  # default factories  is useful in this kind of situation 
# we can also use simple list but this method give us more flixibility in further code . 
    # created_at : datetime = datetime.now(UTC) # it will call datetime once when class is created but we need call every time when a instance is created.
    create_at: datetime = Field(default_factory= partial(datetime.now, tz =UTC) )
    status :Literal['draft', 'pusblished','archiveed'] ='draft'
 

post = BlogPost(
    title ='getting started with python', 
    content='here we go ...',
    author_id='1234',

)
# we can add more annotation ther in the user and blogpost class

print(post)

title='getting started with python' content='here we go ...' author_id='1234' view_count=0 is_published=False tags=[] create_at=datetime.datetime(2026, 6, 24, 11, 19, 54, 547138, tzinfo=datetime.timezone.utc) status='draft'
